# Mushroom Identification Benchmark — Colab Runner

This notebook runs the full comparative benchmark (CNN + Tree + DB + LLM + Unified) with oracle mode on Google Colab.

**Before you start:**
1. Upload `mushroom-benchmark.zip` directly to Colab (drag-and-drop onto the file browser on the left, or use the upload cell below)
2. Runtime → Change runtime type → Select **T4 GPU** (optional but recommended)

**Estimated time:** ~2–4 hours for 57 specimens (with GPU ~1–2 hours)


## Step 1: Upload Project ZIP (if not already uploaded)

If you haven't already dragged `mushroom-benchmark.zip` into the file browser, run this cell to upload it.

In [ ]:
import os
if os.path.exists('/content/mushroom-benchmark.zip'):
    os.remove('/content/mushroom-benchmark.zip')
    print("Removed broken zip")

from google.colab import files
uploaded = files.upload()
for name in uploaded.keys():
    if name.endswith('.zip'):
        os.rename(name, '/content/mushroom-benchmark.zip')
        size_mb = os.path.getsize('/content/mushroom-benchmark.zip') / (1024*1024)
        print(f"Uploaded: {size_mb:.1f} MB")


Removed broken zip


Saving mushroom-benchmark.zip to mushroom-benchmark.zip
Uploaded: 720.7 MB


In [ ]:
import os
from pathlib import Path

print("=== Files in /content/ ===")
for f in sorted(Path('/content/').iterdir(), key=lambda x: x.stat().st_size, reverse=True):
    size_mb = f.stat().st_size / (1024*1024)
    print(f"  {f.name:40s} {size_mb:8.1f} MB")

# Check if any file looks like your zip
print("\n=== Looking for zip-like files ===")
for f in Path('/content/').rglob('*'):
    if f.is_file() and f.suffix in ['.zip', '.tar', '.gz', '.rar', '.7z']:
        size_mb = f.stat().st_size / (1024*1024)
        print(f"  {str(f):50s} {size_mb:8.1f} MB")


=== Files in /content/ ===
  mushroom-benchmark.zip                      565.9 MB
  .config                                       0.0 MB
  mushroom-project                              0.0 MB
  .ipynb_checkpoints                            0.0 MB
  sample_data                                   0.0 MB

=== Looking for zip-like files ===
  /content/mushroom-benchmark.zip                       565.9 MB
  /content/mushroom-project/data/segmentation.zip       154.6 MB
  /content/mushroom-project/data/Yolov8/Manual_annotation/Roboflow_annotation/mushroom segmentation.yolov8(2).zip    161.8 MB
  /content/mushroom-project/data/Yolov8/Manual_annotation/Roboflow_annotation/mushroom_yolo_v2.zip    161.8 MB


In [ ]:
from pathlib import Path

project = Path('/content/mushroom-project')
key_files = [
    'benchmarks/evaluation_manifest_v2.csv',
    'data/Yolov8/best.pt',
    'artifacts/cnn_weights.pt',
    'data/raw/key.xml',
    'data/raw/species_traits.xml',
]

print("=== Checking key files ===")
for f in key_files:
    full = project / f
    if full.exists():
        size_mb = full.stat().st_size / (1024*1024)
        print(f"  ✓ {f} ({size_mb:.1f} MB)")
    else:
        print(f"  ✗ MISSING: {f}")

print(f"
Project root: {project}")
print(f"Exists: {project.exists()}")


=== Checking key files ===
  ✓ benchmarks/evaluation_manifest.csv (0.0 MB)
  ✓ data/Yolov8/best.pt (6.5 MB)
  ✓ artifacts/cnn_weights.pt (41.4 MB)
  ✓ data/raw/key.xml (0.0 MB)
  ✓ data/raw/species_traits.xml (0.1 MB)

Project root: /content/mushroom-project
Exists: True


## Step 2: Extract Project

In [ ]:
import shutil
from pathlib import Path

zip_path = Path('/content/mushroom-benchmark.zip')
project_dir = Path('/content/mushroom-project')

if not zip_path.exists():
    raise FileNotFoundError(f"ZIP not found at {zip_path}. Upload it first.")

# Clean up previous extraction if re-running
if project_dir.exists():
    shutil.rmtree(project_dir)

print("Extracting...")
shutil.unpack_archive(str(zip_path), str(project_dir))
print(f"Project extracted to {project_dir}")

# Verify key files
key_files = [
    'benchmarks/evaluation_manifest_v2.csv',
    'data/Yolov8/best.pt',
    'artifacts/cnn_weights.pt',
    'data/raw/key.xml',
    'data/raw/species_traits.xml',
]
missing = [f for f in key_files if not (project_dir / f).exists()]
if missing:
    print(f"
⚠ Missing: {missing}")
else:
    print("
✓ All key files present.")


Extracting...
Project extracted to /content/mushroom-project

✓ All key files present.


## Step 3: Install Python Dependencies

In [ ]:
%cd /content/mushroom-project

# Core dependencies (skip tensorflow to avoid conflicts with torch)
!pip install -q \
    pandas numpy scikit-learn \
    torch torchvision timm \
    Pillow opencv-python scikit-image \
    h5py tqdm \
    fastapi uvicorn python-multipart \
    pytest pytest-cov \
    python-dotenv pyyaml matplotlib seaborn \
    ultralytics

/content/mushroom-project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.6/254.6 kB 19.4 MB/s eta 0:00:00


## Step 4: Verify GPU (Optional but Recommended)

In [ ]:
!nvidia-smi

Fri May 15 13:08:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 5: Install Ollama

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!nvidia-smi
!which nvidia-smi


Fri May 15 13:08:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 6: Start Ollama Server & Pull Model

In [ ]:
import subprocess
import time
import os

# Kill any existing ollama processes
!pkill -f "ollama serve" 2>/dev/null || true
time.sleep(2)

# Start Ollama in background
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env=env
)

# Wait for server to be ready
print("Starting Ollama server...")
for i in range(30):
    time.sleep(1)
    check = subprocess.run(
        ["curl", "-s", "http://localhost:11434"],
        capture_output=True
    )
    if check.returncode == 0:
        print("Ollama server is ready.")
        break
else:
    print("WARNING: Ollama may not be ready yet.")

^C
Starting Ollama server...
Ollama server is ready.


## Step 7: Pull the Model

In [ ]:
!ollama pull gemma3:12b


## Step 8: Verify Setup

In [ ]:
# Check Ollama is responding and model is loaded
!curl -s http://localhost:11434/api/tags | python3 -m json.tool

# Check project structure
%cd /content/mushroom-project
!ls -la benchmarks/evaluation_manifest.csv data/Yolov8/best.pt artifacts/cnn_weights.pt


{
    "models": [
        {
            "name": "gemma3:4b",
            "model": "gemma3:4b",
            "modified_at": "2026-05-15T13:09:09.547430189Z",
            "size": 3338801804,
            "digest": "a2af6cc3eb7fa8be8504abaf9b04e88f17a119ec3f04a3addf55f92841195f5a",
            "details": {
                "parent_model": "",
                "format": "gguf",
                "family": "gemma3",
                "families": [
                    "gemma3"
                ],
                "parameter_size": "4.3B",
                "quantization_level": "Q4_K_M"
            }
        }
    ]
}
/content/mushroom-project
-rw-r--r-- 1 root root 43426899 May 15 13:07 artifacts/cnn_weights.pt
-rw-r--r-- 1 root root    17540 May 15 13:07 benchmarks/evaluation_manifest.csv
-rw-r--r-- 1 root root  6772788 May 15 13:07 data/Yolov8/best.pt


## Step 9: Keep Colab Alive (Run in Browser Console)

To prevent Colab from disconnecting during the long benchmark run, open your browser's **Developer Tools** (F12), go to the **Console** tab, and paste this code:

```javascript
function ConnectButton(){
    console.log("Keeping Colab alive...");
    document.querySelector("colab-connect-button").click();
}
setInterval(ConnectButton, 60000);
```

This clicks the connect button every 60 seconds.

In [ ]:
# NOTE (May 2025): The patches below are already applied in the current codebase.
# This cell is kept for backward compatibility with older zips but is a no-op.

import os
import sys

FILE_PATH = '/content/mushroom-project/models/llm_classifier.py'
PYCACHE = '/content/mushroom-project/models/__pycache__'

if not os.path.exists(FILE_PATH):
    print("ERROR: File not found. Did you extract the zip?")
else:
    with open(FILE_PATH, "r") as f:
        content = f.read()

    if "import re\n" in content:
        print("✅ Codebase already patched — no action needed.")
    else:
        # Legacy fallback for old zips
        content = content.replace("import os\n", "import os\nimport re\n")
        with open(FILE_PATH, "w") as f:
            f.write(content)
        print("✅ Legacy patch applied: added \"import re\"")

    if os.path.exists(PYCACHE):
        import shutil
        shutil.rmtree(PYCACHE)
        print("✅ Cleared __pycache__")

    if "models.llm_classifier" in sys.modules:
        del sys.modules["models.llm_classifier"]
        print("✅ Cleared module cache")

    sys.path.insert(0, '/content/mushroom-project')
    from models.llm_classifier import LLMClassifier
    print("✅ Module imports successfully")


✅ Added 'import re'
✅ Cleared __pycache__
✅ Module imports successfully
✅ _parse_response works: Test


## Step 10: Run the Benchmark

In [ ]:
import os

os.environ["OLLAMA_TIMEOUT"] = "600"
os.environ["OLLAMA_NUM_PREDICT"] = "512"
os.environ["OLLAMA_BASE_URL"] = "http://localhost:11434"

output_dir = "artifacts/benchmarks/colab_run"
os.makedirs(output_dir, exist_ok=True)

!python3 -m benchmarks.run_comparative \
    --manifest benchmarks/evaluation_manifest_v2.csv \
    --output-dir {output_dir} \
    --variants all


INFO: OracleKeyTree ready: 22 species with paths, 11 oracle, 11 non-oracle
INFO: Oracle species: ['AG.CA', 'AL.OV', 'BO.BA', 'CA.AU', 'CA.TU', 'GO.GL', 'HY.RE', 'LA.VO', 'LY.PE', 'MO.ES', 'SP.CR']
INFO: Non-oracle species: ['AL.CO', 'AR.ME', 'BO.ED', 'CA.CI', 'CR.CO', 'GY.ES', 'HY.RU', 'LE.AU', 'MA.PR', 'RA.BO', 'SU.LU']
INFO: Oracle mode enabled. Oracle species (n=11): AG.CA, AL.OV, BO.BA, CA.AU, CA.TU, GO.GL, HY.RE, LA.VO, LY.PE, MO.ES, SP.CR
INFO: Non-oracle species (n=11): AL.CO, AR.ME, BO.ED, CA.CI, CR.CO, GY.ES, HY.RU, LE.AU, MA.PR, RA.BO, SU.LU
INFO: Loaded 57 specimens from benchmarks/evaluation_manifest.csv
INFO: Initialising CNN runner...
INFO: Loading pretrained weights from Hugging Face hub (timm/efficientnet_b3.ra2_in1k)
INFO: HTTP Request: HEAD https://huggingface.co/timm/efficientnet_b3.ra2_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
INFO: [timm/efficientnet_b3.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading wei

## Step 11: Download Results

In [ ]:
from google.colab import files
import shutil

results_zip = "/content/benchmark_results"
shutil.make_archive(results_zip, 'zip', output_dir)

print("Downloading results...")
files.download(f"{results_zip}.zip")
print("Done!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done!


In [ ]:
from google.colab import files
import shutil

results_dir = "/content/mushroom-project/artifacts/benchmarks/colab_run"
shutil.make_archive("/content/benchmark_results_fixed", 'zip', results_dir)
files.download("/content/benchmark_results_fixed.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# NOTE (May 2025): This patch is already applied in the current codebase.
# Kept for backward compatibility with older zips.

import os

ollama_file = '/content/mushroom-project/models/llm_classifier.py'
with open(ollama_file, "r") as f:
    content = f.read()

if '# "format": "json"' in content or '"format": "json"' not in content:
    print("✅ JSON format handling already correct — no action needed.")
else:
    content = content.replace(
        '"format": "json",   # ask Ollama to enforce JSON output',
        '# "format": "json",  # disabled: gemma3 returns malformed JSON'
    )
    with open(ollama_file, "w") as f:
        f.write(content)
    print("✅ Legacy patch applied: disabled forced JSON mode")

if "def _parse_response" in content and "re.findall" in content:
    print("✅ Robust parser already present")
else:
    print("⚠ Parser may need manual update — re-upload the latest zip")


✓ Patched OllamaBackend (disabled forced JSON mode)
✓ Robust parser already present


In [ ]:
# NOTE (May 2025): This patch is already applied in the current codebase.
# Kept for backward compatibility with older zips.

import os

FILE_PATH = '/content/mushroom-project/models/llm_classifier.py'

if not os.path.exists(FILE_PATH):
    print(f"ERROR: File not found at {FILE_PATH}")
    print("Make sure you ran the extraction cell first.")
else:
    with open(FILE_PATH, "r") as f:
        content = f.read()

    if "import re" in content:
        print("✅ "import re" already present — no fix needed.")
    else:
        # Legacy fallback for old zips
        if "import os
" in content:
            content = content.replace("import os
", "import os
import re
")
            with open(FILE_PATH, "w") as f:
                f.write(content)
            print("✅ Legacy fix applied: added "import re"")
        else:
            print("WARNING: Could not find anchor — fix not applied.")

    # Verify
    with open(FILE_PATH, "r") as f:
        lines = f.readlines()
    for i, line in enumerate(lines[:20], 1):
        if line.strip() == "import re":
            print(f"   Verified at line {i}: {line.rstrip()}")
            break


✅ 'import re' already present — no fix needed.


## Troubleshooting

**Ollama connection refused**
- Make sure Step 6 ran successfully
- Try: `!ollama serve &` instead of the subprocess approach

**Out of disk space**
- `gemma3:12b` is ~8 GB. Check: `!df -h`
- Clear pip cache: `!pip cache purge`

**CUDA/GPU not used by Ollama**
- Ollama auto-detects GPU. If it falls back to CPU, inference will be slower (~same as your laptop)
- Check GPU usage during run: `!nvidia-smi -l 5` in a separate cell

**Session disconnected mid-run**
- Results in `{output_dir}` are lost on disconnect
- Consider running in smaller batches and copying to Drive periodically
